In [1]:
import pandas as pd
import numpy  as np
from transformers import BertTokenizer, BertModel, BertConfig
from transformers import BertForSequenceClassification, Trainer, TrainingArguments ,TrainerCallback
from datasets import load_dataset
import re
from sklearn.metrics import accuracy_score,classification_report,precision_score, recall_score, f1_score
from transformers import pipeline
import torch
from torch.utils.data import DataLoader
import torch.nn as nn


2025-05-03 22:22:19.287134: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746310939.470674      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746310939.525639      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


# BERTIMBAU BASE BINARY

In [ ]:
model_path="./Bertimbau_models/Bert-base-binary"
tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)
model.eval()

# Load the TuPy-E dataset in the multilabel format
ds = load_dataset("Silly-Machine/TuPyE-Dataset", "multilabel")

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+|www.\S+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Final preprocessing function (clean text + tokenize + extract labels)
def preprocess(example):
    cleaned_text = preprocess_text(example["text"])
    enc = tokenizer(cleaned_text, truncation=True, padding="max_length", max_length=128)
    label = int(example["hate"])
    enc["labels"] = [1.0, 0.0] if label == 0 else [0.0, 1.0]

    return enc

tokenized_ds = ds.map(preprocess)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Prepare test dataset
test_dataset = tokenized_ds["test"].with_format("torch")
test_loader = DataLoader(test_dataset, batch_size=32)

# Collect predictions and labels
all_preds = []
all_labels = []

for batch in test_loader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = torch.argmax(batch["labels"], dim=-1).to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1)

    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(labels.cpu().numpy())

# Compute accuracy using sklearn
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(all_labels, all_preds)
print(classification_report(all_labels, all_preds))
print("Test Accuracy :", accuracy)


              precision    recall  f1-score   support

           0       0.93      0.98      0.95      7684
           1       0.75      0.44      0.56      1050

    accuracy                           0.91      8734
   macro avg       0.84      0.71      0.75      8734
weighted avg       0.91      0.91      0.91      8734

Test Accuracy : 0.9149301580032059


# BERTIMBAU LARGE BINARY

In [ ]:
model_path="./Bertimbau_models/Bert-large-binary"
tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)
model.eval()

# Load the TuPy-E dataset in the multilabel format
ds = load_dataset("Silly-Machine/TuPyE-Dataset", "multilabel")

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+|www.\S+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Final preprocessing function (clean text + tokenize + extract labels)
def preprocess(example):
    cleaned_text = preprocess_text(example["text"])
    enc = tokenizer(cleaned_text, truncation=True, padding="max_length", max_length=128)
    label = int(example["hate"])
    enc["labels"] = [1.0, 0.0] if label == 0 else [0.0, 1.0]

    return enc

tokenized_ds = ds.map(preprocess)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Prepare test dataset
test_dataset = tokenized_ds["test"].with_format("torch")
test_loader = DataLoader(test_dataset, batch_size=32)

# Collect predictions and labels
all_preds = []
all_labels = []

for batch in test_loader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = torch.argmax(batch["labels"], dim=-1).to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1)

    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(labels.cpu().numpy())

# Compute accuracy using sklearn
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(all_labels, all_preds)
print(classification_report(all_labels, all_preds))
print("Test Accuracy:", accuracy)


Map:   0%|          | 0/34934 [00:00<?, ? examples/s]

Map:   0%|          | 0/8734 [00:00<?, ? examples/s]

              precision    recall  f1-score   support

           0       0.97      0.97      0.97      7684
           1       0.79      0.78      0.79      1050

    accuracy                           0.95      8734
   macro avg       0.88      0.88      0.88      8734
weighted avg       0.95      0.95      0.95      8734

Test Accuracy: 0.9487062056331578


# BERTIMBAU BASE Hierarchical classification

In [ ]:
model_path="./Bertimbau_models/Bert-base-cat"
tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)
model.eval()

hate_labels = ['ageism', 'aporophobia', 'body_shame', 'capacitism', 'lgbtphobia',
               'political', 'racism', 'religious_intolerance', 'misogyny', 'xenophobia', 'other']

ds = load_dataset("Silly-Machine/TuPyE-Dataset", "multilabel")

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+|www.\S+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Final preprocessing function (clean text + tokenize + extract labels)
def preprocess(example):
    cleaned_text = preprocess_text(example["text"])
    enc = tokenizer(cleaned_text, truncation=True, padding="max_length", max_length=128)
    enc["labels"] = [float(example[label]) for label in hate_labels]
    return enc

tokenized_ds = ds.map(preprocess)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Prepare test dataset (Assuming tokenized_ds is already preprocessed and tokenized)
test_dataset = tokenized_ds["test"].with_format("torch")
test_loader = DataLoader(test_dataset, batch_size=32)

# List to collect all predictions and labels
all_preds = []
all_labels = []


sigmoid = torch.nn.Sigmoid()

# Predict for the entire test set
for batch in test_loader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)
    
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = sigmoid(logits)  # Probabilities for each label (size: [batch_size, num_labels])
        
        preds = (probs >= 0.5).float()  # Convert probabilities to 0 or 1 (multi-label)
    
    # Collect predictions and labels (move them back to CPU for further processing)
    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(labels.cpu().numpy())

# Convert predictions and labels into numpy arrays
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

non_hate_preds = (all_preds.sum(axis=1) == 0).astype(int)
non_hate_true = (all_labels.sum(axis=1) == 0).astype(int)

# Append 'non_hate' as the 12th label (last column)
pred_labels = np.concatenate([all_preds, non_hate_preds[:, None]], axis=1)
true_labels = np.concatenate([all_labels, non_hate_true[:, None]], axis=1)

full_labels = hate_labels + ["non_hate"]

print(classification_report(true_labels, pred_labels, target_names=full_labels, zero_division=0))

Map:   0%|          | 0/34934 [00:00<?, ? examples/s]

Map:   0%|          | 0/8734 [00:00<?, ? examples/s]

                       precision    recall  f1-score   support

               ageism       0.00      0.00      0.00        12
          aporophobia       0.00      0.00      0.00        14
           body_shame       0.88      0.44      0.59        63
           capacitism       0.00      0.00      0.00        11
           lgbtphobia       0.79      0.68      0.73       149
            political       0.63      0.44      0.52       230
               racism       0.65      0.43      0.52        56
religious_intolerance       0.00      0.00      0.00        17
             misogyny       0.71      0.58      0.64       335
           xenophobia       0.67      0.27      0.39        88
                other       0.56      0.36      0.44       879
             non_hate       0.90      0.95      0.92      7188

            micro avg       0.86      0.85      0.85      9042
            macro avg       0.48      0.35      0.40      9042
         weighted avg       0.84      0.85      0.84 

# BERTIMBAU LARGE Hierarchical classification

In [ ]:
model_path="./Bertimbau_models/Bert-large-cat"
tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)
model.eval()

hate_labels = ['ageism', 'aporophobia', 'body_shame', 'capacitism', 'lgbtphobia',
               'political', 'racism', 'religious_intolerance', 'misogyny', 'xenophobia', 'other']

ds = load_dataset("Silly-Machine/TuPyE-Dataset", "multilabel")

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+|www.\S+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Final preprocessing function (clean text + tokenize + extract labels)
def preprocess(example):
    cleaned_text = preprocess_text(example["text"])
    enc = tokenizer(cleaned_text, truncation=True, padding="max_length", max_length=128)
    enc["labels"] = [float(example[label]) for label in hate_labels]
    return enc

tokenized_ds = ds.map(preprocess)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Prepare test dataset (Assuming tokenized_ds is already preprocessed and tokenized)
test_dataset = tokenized_ds["test"].with_format("torch")
test_loader = DataLoader(test_dataset, batch_size=32)

# List to collect all predictions and labels
all_preds = []
all_labels = []

sigmoid = torch.nn.Sigmoid()

# Predict for the entire test set
for batch in test_loader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)
    
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = sigmoid(logits)  # Probabilities for each label (size: [batch_size, num_labels])
        
        preds = (probs >= 0.5).float()  # Convert probabilities to 0 or 1 (multi-label)
    
    # Collect predictions and labels (move them back to CPU for further processing)
    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(labels.cpu().numpy())

# Convert predictions and labels into numpy arrays
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

non_hate_preds = (all_preds.sum(axis=1) == 0).astype(int)
non_hate_true = (all_labels.sum(axis=1) == 0).astype(int)

# Append 'non_hate' as the 12th label (last column)
pred_labels = np.concatenate([all_preds, non_hate_preds[:, None]], axis=1)
true_labels = np.concatenate([all_labels, non_hate_true[:, None]], axis=1)

full_labels = hate_labels + ["non_hate"]

print(classification_report(true_labels, pred_labels, target_names=full_labels, zero_division=0))

Map:   0%|          | 0/34934 [00:00<?, ? examples/s]

Map:   0%|          | 0/8734 [00:00<?, ? examples/s]

                       precision    recall  f1-score   support

               ageism       0.00      0.00      0.00        12
          aporophobia       0.00      0.00      0.00        14
           body_shame       0.82      0.52      0.64        63
           capacitism       0.00      0.00      0.00        11
           lgbtphobia       0.80      0.63      0.70       149
            political       0.78      0.24      0.37       230
               racism       0.50      0.07      0.12        56
religious_intolerance       0.00      0.00      0.00        17
             misogyny       0.66      0.64      0.65       335
           xenophobia       0.53      0.09      0.16        88
                other       0.61      0.47      0.53       879
             non_hate       0.90      0.96      0.93      7188

            micro avg       0.86      0.85      0.86      9042
            macro avg       0.47      0.30      0.34      9042
         weighted avg       0.84      0.85      0.84 

# roBERTa BASE Binary

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_path = "./Improved_models/Roberta-base-binary"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path,num_labels=2)

# Put the model in evaluation mode
model.eval()

# Load the TuPy-E dataset in the multilabel format
ds = load_dataset("Silly-Machine/TuPyE-Dataset", "multilabel")

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+|www.\S+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Final preprocessing function (clean text + tokenize + extract labels)
def preprocess(example):
    cleaned_text = preprocess_text(example["text"])
    enc = tokenizer(cleaned_text, truncation=True, padding="max_length", max_length=128)
    label = int(example["hate"])
    enc["labels"] = [1.0, 0.0] if label == 0 else [0.0, 1.0]

    return enc

tokenized_ds = ds.map(preprocess)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Prepare test dataset
test_dataset = tokenized_ds["test"].with_format("torch")
test_loader = DataLoader(test_dataset, batch_size=32)

# Collect predictions and labels
all_preds = []
all_labels = []

for batch in test_loader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = torch.argmax(batch["labels"], dim=-1).to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1)

    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(labels.cpu().numpy())

# Compute accuracy using sklearn
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(all_labels, all_preds)
print(classification_report(all_labels, all_preds))
print("Test Accuracy:", accuracy)


Map:   0%|          | 0/34934 [00:00<?, ? examples/s]

Map:   0%|          | 0/8734 [00:00<?, ? examples/s]

              precision    recall  f1-score   support

           0       0.94      0.98      0.96      7684
           1       0.76      0.56      0.64      1050

    accuracy                           0.93      8734
   macro avg       0.85      0.77      0.80      8734
weighted avg       0.92      0.93      0.92      8734

Test Accuracy: 0.9258071902908175


In [9]:
class BertWithCNN(nn.Module):
    def __init__(self, model_path, num_labels=2, freeze_bert=True):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_path)

        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=self.bert.config.hidden_size, out_channels=128, kernel_size=k)
            for k in [2, 3, 4]
        ])
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(128 * len(self.convs), num_labels)

        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        x = outputs.last_hidden_state        # [B, L, H]
        x = x.permute(0, 2, 1)               # [B, H, L]

        x = [torch.relu(conv(x)) for conv in self.convs]
        x = [torch.max(conv, dim=2)[0] for conv in x]
        x = torch.cat(x, dim=1)              # [B, 128 * 3]

        x = self.dropout(x)
        logits = self.classifier(x)
        if labels is not None:
            loss_fn = nn.BCEWithLogitsLoss()
            loss = loss_fn(logits, labels)
            return {"loss": loss, "logits": logits}
        return {"logits": logits}


# CNN Model Binary

In [ ]:
model_path="./Improved_models/CNN_model_binary"
tokenizer = BertTokenizer.from_pretrained(model_path)
model = torch.load('./Improved_models/CNN_model_binary/CNN_model.pth',weights_only=False)
model.eval()

# Load the TuPy-E dataset in the multilabel format
ds = load_dataset("Silly-Machine/TuPyE-Dataset", "multilabel")

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+|www.\S+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Final preprocessing function (clean text + tokenize + extract labels)
def preprocess(example):
    cleaned_text = preprocess_text(example["text"])
    enc = tokenizer(cleaned_text, truncation=True, padding="max_length", max_length=128)
    label = int(example["hate"])
    enc["labels"] = [1.0, 0.0] if label == 0 else [0.0, 1.0]

    return enc

tokenized_ds = ds.map(preprocess)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Prepare test dataset
test_dataset = tokenized_ds["test"].with_format("torch")
test_loader = DataLoader(test_dataset, batch_size=32)

# Collect predictions and labels
all_preds = []
all_labels = []

for batch in test_loader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = torch.argmax(batch["labels"], dim=-1).to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs["logits"]
        preds = torch.argmax(logits, dim=-1)

    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(labels.cpu().numpy())

# Compute accuracy using sklearn
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(all_labels, all_preds)
print(classification_report(all_labels, all_preds))
print("Test Accuracy :", accuracy)


Map:   0%|          | 0/34934 [00:00<?, ? examples/s]

Map:   0%|          | 0/8734 [00:00<?, ? examples/s]

              precision    recall  f1-score   support

           0       0.94      0.97      0.96      7684
           1       0.73      0.54      0.62      1050

    accuracy                           0.92      8734
   macro avg       0.83      0.75      0.79      8734
weighted avg       0.91      0.92      0.91      8734

Test Accuracy : 0.9203114266086558


# CNN with unfreezing BERT-base layers Binary

In [ ]:
model_path="./Improved_models/CNN_grad_unfreezing_bert_base_binary"
tokenizer = BertTokenizer.from_pretrained(model_path)
model = torch.load('./Improved_models/CNN_grad_unfreezing_bert_base_binary/CNN_grad_unfreeze_bert.pth',weights_only=False)
model.eval()

# Load the TuPy-E dataset in the multilabel format
ds = load_dataset("Silly-Machine/TuPyE-Dataset", "multilabel")

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+|www.\S+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Final preprocessing function (clean text + tokenize + extract labels)
def preprocess(example):
    cleaned_text = preprocess_text(example["text"])
    enc = tokenizer(cleaned_text, truncation=True, padding="max_length", max_length=128)
    label = int(example["hate"])
    enc["labels"] = [1.0, 0.0] if label == 0 else [0.0, 1.0]

    return enc

tokenized_ds = ds.map(preprocess)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Prepare test dataset
test_dataset = tokenized_ds["test"].with_format("torch")
test_loader = DataLoader(test_dataset, batch_size=32)

# Collect predictions and labels
all_preds = []
all_labels = []

for batch in test_loader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = torch.argmax(batch["labels"], dim=-1).to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs["logits"]
        preds = torch.argmax(logits, dim=-1)

    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(labels.cpu().numpy())

# Compute accuracy using sklearn
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(all_labels, all_preds)
print(classification_report(all_labels, all_preds))
print("Test Accuracy :", accuracy)


Map:   0%|          | 0/34934 [00:00<?, ? examples/s]

Map:   0%|          | 0/8734 [00:00<?, ? examples/s]

              precision    recall  f1-score   support

           0       0.94      0.98      0.96      7684
           1       0.74      0.52      0.61      1050

    accuracy                           0.92      8734
   macro avg       0.84      0.75      0.78      8734
weighted avg       0.91      0.92      0.91      8734

Test Accuracy : 0.9205404167620792


# CNN with unfreezing BERT-large layers Binary

In [ ]:
model_path="./Improved_models/CNN_grad_unfreezing_bert_large_binary"
tokenizer = BertTokenizer.from_pretrained(model_path)
model = torch.load('./Improved_models/CNN_grad_unfreezing_bert_large_binary/CNN_grad_unfreeze_bertl.pth',weights_only=False)
model.eval()

# Load the TuPy-E dataset in the multilabel format
ds = load_dataset("Silly-Machine/TuPyE-Dataset", "multilabel")

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+|www.\S+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Final preprocessing function (clean text + tokenize + extract labels)
def preprocess(example):
    cleaned_text = preprocess_text(example["text"])
    enc = tokenizer(cleaned_text, truncation=True, padding="max_length", max_length=128)
    label = int(example["hate"])
    enc["labels"] = [1.0, 0.0] if label == 0 else [0.0, 1.0]

    return enc

tokenized_ds = ds.map(preprocess)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Prepare test dataset
test_dataset = tokenized_ds["test"].with_format("torch")
test_loader = DataLoader(test_dataset, batch_size=32)

# Collect predictions and labels
all_preds = []
all_labels = []

for batch in test_loader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = torch.argmax(batch["labels"], dim=-1).to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs["logits"]
        preds = torch.argmax(logits, dim=-1)

    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(labels.cpu().numpy())

# Compute accuracy using sklearn
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(all_labels, all_preds)
print(classification_report(all_labels, all_preds))
print("Test Accuracy :", accuracy)


Map:   0%|          | 0/34934 [00:00<?, ? examples/s]

Map:   0%|          | 0/8734 [00:00<?, ? examples/s]

              precision    recall  f1-score   support

           0       0.96      0.98      0.97      7684
           1       0.84      0.74      0.78      1050

    accuracy                           0.95      8734
   macro avg       0.90      0.86      0.88      8734
weighted avg       0.95      0.95      0.95      8734

Test Accuracy : 0.9511106022441035
